# 50 — Gold fact: `fact_exposure_snapshot`
**Grain:** one row per LC × exposure type × `as_of_date` (open exposure only: amount > 0).
**Snapshot measure** — never sum across dates.

Re-running for the same `as_of_date` replaces that date only (idempotent).

In [ ]:
%run ./_common

In [ ]:
FACT = fq("gold", "fact_exposure_snapshot")
asof = F.lit(AS_OF).cast("date")
asof_sk = int(AS_OF.replace("-", ""))

# 1. latest exposure event per LC and exposure type, as of end of the business date
events = spark.table(fq("silver", "lc_exposure_event")).where(F.col("occurred_at") < F.date_add(asof, 1).cast("timestamp"))
w = Window.partitionBy("lc_id", "exposure_type").orderBy(F.col("occurred_at").desc(), F.col("event_id").desc())
latest = (events.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")
          .where("exposure_amount > 0")
          .select("lc_id", "exposure_type", "lifecycle_status", F.col("exposure_amount").alias("outstanding_exposure_local"),
                  F.col("currency").alias("exposure_currency")))

# 2. LC attributes + issuing bank, only LCs that exist as of the date
lc = spark.table(fq("silver", "letter_of_credit")).where(F.col("issue_date") <= asof) \
          .select("lc_id", "issuing_bank_id", "amount")

# 3. current LC amount = original + approved amendments effective by the date
amend = (spark.table(fq("silver", "lc_amendment"))
         .where((F.col("approval_status") == "APPROVED") & (F.col("effective_at") < F.date_add(asof, 1).cast("timestamp")))
         .groupBy("lc_id").agg(F.sum("amount_delta").alias("amount_delta")))

# 4. applicant valid on the date
role = (spark.table(fq("silver", "lc_party_role"))
        .where((F.col("role") == "APPLICANT") & (F.col("valid_from") <= asof) & (F.col("valid_to").isNull() | (F.col("valid_to") >= asof)))
        .withColumn("_rn", F.row_number().over(Window.partitionBy("lc_id").orderBy(F.col("valid_from").desc())))
        .where("_rn = 1").select("lc_id", F.col("party_id").alias("applicant_party_id")))

# 5. dimension keys (SCD2 counterparty: version in effect on the date)
dim_cp = spark.table(fq("gold", "dim_counterparty")).where((F.col("eff_start_date") <= asof) & (F.col("eff_end_date") > asof)) \
              .select(F.col("party_id").alias("applicant_party_id"), "counterparty_sk")
dim_cur = spark.table(fq("gold", "dim_currency")).select("currency_sk", F.col("currency_code").alias("exposure_currency"),
                                                        F.col("rate_to_usd").alias("fx_rate_to_usd"))

fact = (latest.join(lc, "lc_id", "inner")
        .join(amend, "lc_id", "left")
        .join(role, "lc_id", "left")
        .join(dim_cp, "applicant_party_id", "left")
        .join(dim_cur, "exposure_currency", "left")
        .select(
            sk("lc_id", "exposure_type", F.lit(AS_OF)).alias("exposure_sk"),
            F.lit(asof_sk).alias("as_of_date_sk"),
            sk("lc_id").alias("lc_sk"),
            F.coalesce("counterparty_sk", F.lit(-1).cast("long")).alias("counterparty_sk"),
            sk("issuing_bank_id").alias("bank_sk"),
            F.coalesce("currency_sk", F.lit(-1).cast("long")).alias("currency_sk"),
            "exposure_type", "lifecycle_status",
            F.col("outstanding_exposure_local").cast("decimal(18,2)"),
            (F.col("outstanding_exposure_local") * F.col("fx_rate_to_usd")).cast("decimal(18,2)").alias("outstanding_exposure_usd"),
            (F.col("amount") + F.coalesce("amount_delta", F.lit(0))).cast("decimal(18,2)").alias("lc_amount_current_local"),
            F.col("fx_rate_to_usd").cast("decimal(18,8)"),
            F.current_timestamp().alias("_loaded_at")))

if spark.catalog.tableExists(FACT):
    fact.write.format("delta").mode("overwrite").option("replaceWhere", f"as_of_date_sk = {asof_sk}").saveAsTable(FACT)
else:
    fact.write.format("delta").partitionBy("as_of_date_sk").saveAsTable(FACT)
print(f"fact rows for {AS_OF}:", spark.table(FACT).where(f"as_of_date_sk = {asof_sk}").count())

In [ ]:
# Business view for the outcome: exposure per counterparty per day, rating as at that day
spark.sql(f"""
CREATE OR REPLACE VIEW {fq('gold', 'v_counterparty_exposure_daily')} AS
SELECT d.date AS as_of_date,
       c.party_id AS counterparty_id,
       c.counterparty_name,
       c.risk_rating,
       c.country AS counterparty_country,
       COUNT(DISTINCT f.lc_sk) AS open_lc_count,
       SUM(f.outstanding_exposure_usd) AS outstanding_exposure_usd
FROM {fq('gold', 'fact_exposure_snapshot')} f
JOIN {fq('gold', 'dim_date')} d ON d.date_sk = f.as_of_date_sk
JOIN {fq('gold', 'dim_counterparty')} c ON c.counterparty_sk = f.counterparty_sk
GROUP BY d.date, c.party_id, c.counterparty_name, c.risk_rating, c.country
""")

In [ ]:
display(spark.table(fq("gold", "v_counterparty_exposure_daily")).where(F.col("as_of_date") == asof)
        .orderBy(F.col("outstanding_exposure_usd").desc()).limit(10))